### ChromaDB, FAISS, MMilvus

#### Creation of Vector Store 

Lets get started with chormaDB

Open source search and retrival database for AI Application


## Building A RAG system with Langchain and ChromaDB

introduction 

Retrival - Augmented generation(RAG) is a powerfull technique that combines the capabilities of large language models with external knowlege retrivslo.THis notebook will walk you through building a complete RAG system using:

** Langchian:A Framework for developijng application powered by language models 
** Chroma DB L: An open source Vector database for storing and retriving Embeddings 
** OpenAI : for embeddings and language model (you can substitute with other Providers)


In [5]:
import os 
from dotenv import load_dotenv
load_dotenv()

True

In [6]:

from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import TextLoader
from langchain_openai import OpenAIEmbeddings
from langchain.schema import Document


### Retrival-Augmented generation)Architecture:

1. Document Loading: Load Documents from various Sources 
2. Document Splitting : Break documents into smalller parts
3. embeddings generation: convert chunks into vector representations
4. vector stroage:store embeddings in chromaDB 
5. Query Provessing: Convert user query to embeddings
6. imilarity search:find relevent chunks from vector stroage
7. ontext Augmentation: Combine retrived chunks with query 
8. sponse generation : LLM generates answer using Context



In [7]:
### Create some kinda of sample data 

sample_docs =[
    """Machine Learning is a subset of artificial intelligence that focuses on the development of algorithms and statistical models that enable computers to perform tasks without explicit instructions.""",
    """Reinforcement Learning is an area of machine learning concerned with how agents ought to take actions in an environment in order to maximize some notion of cumulative reward.""",
    """Natural Language Processing (NLP) is a field of artificial intelligence that focuses on the interaction between computers and humans through natural language.""",
    """Deep Learning is a subset of machine learning that uses neural networks with many layers (hence "deep") to analyze various types of data.""",
    """Machine Learning is a rapidly evolving field, and staying up-to-date with the latest research and techniques is crucial for practitioners."""
]
sample_docs

['Machine Learning is a subset of artificial intelligence that focuses on the development of algorithms and statistical models that enable computers to perform tasks without explicit instructions.',
 'Reinforcement Learning is an area of machine learning concerned with how agents ought to take actions in an environment in order to maximize some notion of cumulative reward.',
 'Natural Language Processing (NLP) is a field of artificial intelligence that focuses on the interaction between computers and humans through natural language.',
 'Deep Learning is a subset of machine learning that uses neural networks with many layers (hence "deep") to analyze various types of data.',
 'Machine Learning is a rapidly evolving field, and staying up-to-date with the latest research and techniques is crucial for practitioners.']

In [8]:
### save the sample Documents to txt files

import tempfile
temp_dir=tempfile.mkdtemp()

for i ,doc in enumerate(sample_docs):
    with open(os.path.join(temp_dir,f"doc_{i}.txt"),"w") as f:
        f.write(doc)
print(f"Sample documents saved in {temp_dir}")

Sample documents saved in C:\Users\siddv\AppData\Local\Temp\tmpbg1tp_d9


### 2. Document loading 


In [9]:
from langchain_community.document_loaders import DirectoryLoader,TextLoader

load =DirectoryLoader(
    temp_dir,
    glob="*.txt",
    loader_cls=TextLoader,
    loader_kwargs={"encoding":"utf-8"}
)
documents = load.load()

print(f"loaded {len(documents)} documents")
print(documents[0].page_content[:200] + "...")  

loaded 5 documents
Machine Learning is a subset of artificial intelligence that focuses on the development of algorithms and statistical models that enable computers to perform tasks without explicit instructions....


### Document Split



In [10]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=100,
    length_function=len,
    separators=[""]
)

chunks=text_splitter.split_documents(documents)

print(f"Created {len(chunks)} chunks from {len(documents)} documents")

Created 5 chunks from 5 documents


In [11]:
chunks

[Document(metadata={'source': 'C:\\Users\\siddv\\AppData\\Local\\Temp\\tmpbg1tp_d9\\doc_0.txt'}, page_content='Machine Learning is a subset of artificial intelligence that focuses on the development of algorithms and statistical models that enable computers to perform tasks without explicit instructions.'),
 Document(metadata={'source': 'C:\\Users\\siddv\\AppData\\Local\\Temp\\tmpbg1tp_d9\\doc_1.txt'}, page_content='Reinforcement Learning is an area of machine learning concerned with how agents ought to take actions in an environment in order to maximize some notion of cumulative reward.'),
 Document(metadata={'source': 'C:\\Users\\siddv\\AppData\\Local\\Temp\\tmpbg1tp_d9\\doc_2.txt'}, page_content='Natural Language Processing (NLP) is a field of artificial intelligence that focuses on the interaction between computers and humans through natural language.'),
 Document(metadata={'source': 'C:\\Users\\siddv\\AppData\\Local\\Temp\\tmpbg1tp_d9\\doc_3.txt'}, page_content='Deep Learning is a

### Apply embedding Models


In [12]:
from langchain.embeddings import HuggingFaceEmbeddings
sample_text = "This is a sample text for embedding."

embeddings=HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")   
embeddings

C:\Users\siddv\AppData\Local\Temp\ipykernel_23600\1905924878.py:4: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings=HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
d:\RAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


HuggingFaceEmbeddings(client=SentenceTransformer(
  (0): Transformer({'max_seq_length': 256, 'do_lower_case': False, 'architecture': 'BertModel'})
  (1): Pooling({'word_embedding_dimension': 384, 'pooling_mode_cls_token': False, 'pooling_mode_mean_tokens': True, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False, 'pooling_mode_weightedmean_tokens': False, 'pooling_mode_lasttoken': False, 'include_prompt': True})
  (2): Normalize()
), model_name='sentence-transformers/all-MiniLM-L6-v2', cache_folder=None, model_kwargs={}, encode_kwargs={}, multi_process=False, show_progress=False)

In [13]:
vector= embeddings.embed_query(sample_text)

In [14]:
# Get embeddings for each document chunk
vectors = [embeddings.embed_query(chunk.page_content) for chunk in chunks]
vectors

[[-0.005222750827670097,
  -0.0016360229346901178,
  0.06075487285852432,
  0.02015822008252144,
  -0.0120776928961277,
  -0.035130131989717484,
  -0.040626462548971176,
  -0.04060107097029686,
  -0.06481927633285522,
  -0.015857333317399025,
  -0.05414237082004547,
  -0.0008518153917975724,
  0.0332816019654274,
  -0.07091318070888519,
  -0.019978223368525505,
  0.01628214679658413,
  0.020561255514621735,
  -0.01189093291759491,
  -0.07532045990228653,
  -0.06647171825170517,
  0.06209978461265564,
  -0.009791925549507141,
  -0.05864831805229187,
  0.0319051668047905,
  -0.007730367127805948,
  0.020192356780171394,
  0.05991419032216072,
  0.044153615832328796,
  -0.027278045192360878,
  0.04034247621893883,
  0.023821691051125526,
  -0.05459721013903618,
  0.041888926178216934,
  0.028389884158968925,
  -0.04214039072394371,
  0.01914246194064617,
  -0.07208047807216644,
  0.0312851220369339,
  0.013528489507734776,
  -0.013212187215685844,
  -0.05448073148727417,
  -0.050361011177

### intialize the chroma DB vector Store and store thje chunks in Vector Represenations

In [15]:
from langchain_community.vectorstores import Chroma

### Create a chroma Db vector Store 

persist_directory= "./chroma_db"

## Initialize chromadb with HuggingFaceEmbeddings
vector_store = Chroma.from_documents(
	documents=chunks,
	embedding=embeddings,  # Changed 'embeddings' to 'embedding'
	persist_directory=persist_directory,
	collection_name="RAG_collection"
)

print(f"vector store created with {vector_store._collection.count()} vectors")
print(f"persisted to {persist_directory}")

vector store created with 10 vectors
persisted to ./chroma_db


In [16]:
### test Similarity Search
query = "What are the types of Deep learning?"
results = vector_store.similarity_search(query,k=2)
results
    

[Document(metadata={'source': 'C:\\Users\\siddv\\AppData\\Local\\Temp\\tmpll925vza\\doc_3.txt'}, page_content='Deep Learning is a subset of machine learning that uses neural networks with many layers (hence "deep") to analyze various types of data.'),
 Document(metadata={'source': 'C:\\Users\\siddv\\AppData\\Local\\Temp\\tmpbg1tp_d9\\doc_3.txt'}, page_content='Deep Learning is a subset of machine learning that uses neural networks with many layers (hence "deep") to analyze various types of data.')]

In [17]:
### Advanced Similarity Search with Scores

results_scores=vector_store.similarity_search_with_score(query,k=2)
results_scores

[(Document(metadata={'source': 'C:\\Users\\siddv\\AppData\\Local\\Temp\\tmpll925vza\\doc_3.txt'}, page_content='Deep Learning is a subset of machine learning that uses neural networks with many layers (hence "deep") to analyze various types of data.'),
  0.5363179445266724),
 (Document(metadata={'source': 'C:\\Users\\siddv\\AppData\\Local\\Temp\\tmpbg1tp_d9\\doc_3.txt'}, page_content='Deep Learning is a subset of machine learning that uses neural networks with many layers (hence "deep") to analyze various types of data.'),
  0.5363179445266724)]

### 2. understanding Similairty Scores

the similarity SCore represents how closely related a document chunk is to your query .The scoring depends n the distance metric used:

  chromaDB default uses L2 distance(Eucliden Diastance)

1. ower Scores = More Similar(closer in vector space)
2. score of 0 = identical vectors
3. Typical range: 0 to 2 but can vary based on the dataset and query complexity

Cosine Similarity 

1. higher Score(if configured)
2. range -1 to 1 (1 being identical)

#### initilize the LLM, RAG chain, PRompt TEmplete , Query the RAG system

In [18]:
from langchain_openai import ChatOpenAI
llm = ChatOpenAI(
    model="gpt-3.5-turbo",
    temperature=0.2,
    )

In [19]:
test_response=llm.invoke("what is LLM?")
test_response

RateLimitError: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}

In [ ]:
from langchain.chat_models.base import init_chat_model 

LLM = init_chat_model("openai:spt-3.5-turbo")

In [20]:
llm

ChatOpenAI(client=<openai.resources.chat.completions.completions.Completions object at 0x00000252821BCF90>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x000002528E955810>, root_client=<openai.OpenAI object at 0x00000252D96A2B10>, root_async_client=<openai.AsyncOpenAI object at 0x000002528E955510>, temperature=0.2, model_kwargs={}, openai_api_key=SecretStr('**********'))

### Modern RAG Chain

##### Traditional RAG : Fuction :- Create_Retrival_chain is a function in langchian which take care of retrival and augmentation and helps in connecting with LLM 

ChatPromptTemplate, 

converting vector store to retriver so that i should be included in create_retrival_chain using  .as_retriver inbuilt function it acts as the interface for retrival of documents 

Prompt templete is needed coz whatever the retrived data needs to interact with LLM 

In [21]:
## when we talk about the retrival chain , create_retrival_chain will create pipline from context we are getting to the LLm and finally getting the output 

from langchain.chains import create_retrieval_chain
from langchain_core.prompts import ChatPromptTemplate
from langchain.chains.combine_documents import create_stuff_documents_chain


In [ ]:
## Convert vector store to retriver

retriever = vector_store.as_retriever(
    search_kwargs={"k":3}
)
retriever

NameError: name 'vector_store' is not defined

In [22]:
## create a prompt Template 
from langchain_core.prompts import ChatPromptTemplate
system_prompt = """you are an assitant for question answering tasks. use the following context to answer the questions: {context}

context:{context}"""

prompt = ChatPromptTemplate.from_messages([(
    "system",system_prompt),
    ("human","(input)")
    ])

In [23]:
prompt

ChatPromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template='you are an assitant for question answering tasks. use the following context to answer the questions: {context}\n\ncontext:{context}'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='(input)'), additional_kwargs={})])

In [24]:
### Create a document chain 
###what doc change is all about?
#  Create_stuff_documents_chain : whatever relevent information from vector store it will combine and give to LLM , 
from langchain.chains.combine_documents import create_stuff_documents_chain

document_chain = create_stuff_documents_chain(llm=llm, prompt=prompt)
document_chain

### takes retrived document
## "stuffs" them into the prompt's {context} placeholder
## send the complete prompot to the LLM 
### Returns the LLM response 

RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableLambda(format_docs)
}), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
| ChatPromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template='you are an assitant for question answering tasks. use the following context to answer the questions: {context}\n\ncontext:{context}'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='(input)'), additional_kwargs={})])
| ChatOpenAI(client=<openai.resources.chat.completions.completions.Completions object at 0x00000252821BCF90>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x000002528E955810>, root_client=<openai.OpenAI object at 0x00000252D96A2B10>, root_asy

In [25]:
### Create the final RAG chain
from langchain.chains import create_retrieval_chain
rag_chain = create_retrieval_chain(retriever,document_chain)
rag_chain 

NameError: name 'retriever' is not defined

In [26]:
rag_chain.invoke("What are the types of Machine Learning?")

NameError: name 'rag_chain' is not defined

In [27]:
response['answer']

NameError: name 'response' is not defined